# Autograd & requires_grad

Every tensor has a `requires_grad` flag. When it is `True`, PyTorch tracks the operations done with it.

- Any operation with at least one tracked tensor produces a tracked result.
- Each such operation creates a **node** in a graph (`grad_fn`), which stores:
  - the **derivative formula** of that operation,
  - **pointers** to its inputs,
  - the **forward values** the derivative needs (saved tensors).
- The graph lives on the tensors themselves and is rebuilt at every forward pass.

In [22]:
import torch

a = torch.tensor(2.0, requires_grad = True)

x = a ** 2
z = x * 2

print(z.grad_fn.next_functions)

((<PowBackward0 object at 0x00000203461DEC50>, 0), (None, 0))


# Backward

After the forward pass builds the graph, calling `backward()` on the loss walks it **from the output back to the inputs**. At each node, it applies that operation's derivative formula and passes the result to the node's inputs (chain rule).

The final values are stored in the `.grad` attribute of the **leaf tensors** (e.g. model parameters), and each one tells how that tensor should change to reduce the loss.

In [23]:
z.backward()

print(a.grad)

tensor(8.)


# Gradient Accumulation (BACKWARD)

Each `backward()` call **adds** the new gradient to the value already stored in `.grad`, instead of overwriting it.

So if `backward()` runs multiple times, the gradients **accumulate over time**:

backward 1 → .grad = g1
backward 2 → .grad = g1 + g2
backward 3 → .grad = g1 + g2 + g3


This also happens inside a single graph: when a tensor is used in several places, the gradients from each path are summed.

It can be used on purpose (to simulate a larger batch, summing several micro-batches before updating), but in normal training the old gradient must be cleared every step with `zero_grad()`, otherwise stale gradients leak into the update.

In [25]:
w = torch.tensor(3.0, requires_grad=True)

loss = w * 2
loss.backward()
print(w.grad)    # tensor(2.)   → era None, então guarda 2

loss = w * 5
loss.backward()
print(w.grad)    # tensor(7.)   → 2 + 5, somou

tensor(2.)
tensor(7.)
